# Benchmark: FLUX.2 klein 4B on T4 16GB (local model)
Measures load time, generation time, peak VRAM. Set Runtime to T4 GPU first.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf

## Log in to Hugging Face
Accept access at huggingface.co/black-forest-labs/FLUX.2-klein-4B first if gated.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch, time
from diffusers import Flux2KleinPipeline

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
load_start = time.time()
pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
pipe.enable_model_cpu_offload()
load_time = time.time() - load_start
print(f"Model loaded in {load_time:.1f}s")

In [ ]:
PROMPT = ("Chiku and Pinku starting their adventure in the garden. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Chiku, the small sleek cat with soft gray fur. Pointed ears. Sharp green eyes. Long graceful tail. Four agile legs. Standing on grass. Pinku, the friendly dog with golden brown fur. Floppy ears. Bright loyal eyes. Wagging tail. Four legs. Running beside Chiku. Both characters visible. Jungle garden setting with lush green grass and bushes. Afternoon sunlight filtering through leaves. Light beige ground. The scene shows excitement and adventure.")

torch.cuda.reset_peak_memory_stats()
gen_start = time.time()
image = pipe(prompt=PROMPT, height=1024, width=1024, num_inference_steps=4, guidance_scale=1.0, generator=torch.Generator(device="cuda").manual_seed(42)).images[0]
gen_time = time.time() - gen_start
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

image.save("benchmark_klein4b.png")
print(f"Load: {load_time:.1f}s | Generation: {gen_time:.2f}s | Peak VRAM: {peak_vram_gb:.2f} GB")
image